# Layer-wise Encoder Feature Cosine Distance (BCIC 2A)

`layer_wise_feature_distance_BCIC2A.py`가 slicing_mode/feature_mode 조합별로 저장한 4개의 CSV를 불러와서,

- x축: layer
- y축: 각 metric 값 (`same_index_pairwise_mean`, `full_pairwise_mean`, `mean_vector_distance`)
- 한 그래프 안에 5개 method(linear_probe, finetune, LORA, VERA, dynamic_pearl)를 겹쳐서 표시

형태로 그립니다. 아래 `SPLIT_TO_PLOT` 값만 `'train'` / `'valid'` / `'test'`로 바꾸면 해당 split의 그래프를 그릴 수 있습니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 0. 설정값
# ==========================================
LOG_DIR = "/workspace/EEGPT/log/BCIC2A_log"
CSV_DATE_TAG = "260926"

# 예: 260926_layer_wise_feature_distance_raw_flatten.csv
def csv_path(slicing_mode, feature_mode):
    return (
        f"{LOG_DIR}/{CSV_DATE_TAG}_layer_wise_feature_distance_"
        f"{slicing_mode}_{feature_mode}.csv"
    )

# (slicing_mode, feature_mode) 4가지 조합
COMBOS = [
    ("raw", "flatten"),
    ("raw", "pooling"),
    ("sliced", "flatten"),
    ("sliced", "pooling"),
]

# 각각 별도의 그래프로 그릴 metric 목록
METRICS = [
    "same_index_pairwise_mean",
    "full_pairwise_mean",
    "mean_vector_distance",
]

# 지금은 test set만, 나중에 'train' 또는 'valid'로 바꿔서 재실행하면 됨
SPLIT_TO_PLOT = "test"

# CSV의 method 값 -> 범례에 표시할 라벨 매핑 (frozen은 reference라 그래프에서 제외)
METHOD_LABELS = {
    "linear_probe": "Linear probe",
    "finetune": "Fine tuning",
    "LORA": "LORA",
    "VERA": "VERA",
    "dynamic_pearl": "Dynamic Pearl",
}

METHOD_STYLES = {
    "Linear probe":  {"color": "#2A78D6", "linestyle": (0, (3, 1, 1, 1)), "marker": "v"},
    "VERA":          {"color": "#EDA100", "linestyle": "--",             "marker": "s"},
    "Pearl":         {"color": "#1BAF7A", "linestyle": "-.",             "marker": "^"},
    "Fine tuning":   {"color": "#4A3AA7", "linestyle": ":",              "marker": "D"},
    "Dynamic Pearl": {"color": "#E34948", "linestyle": "-",              "marker": "o"},
    "LORA":          {"color": "#008300", "linestyle": (0, (5, 1)),      "marker": "P"},
}

In [ ]:
# ==========================================
# 1. CSV 로드
# ==========================================
def load_distance_csv(slicing_mode, feature_mode):
    path = csv_path(slicing_mode, feature_mode)
    df = pd.read_csv(path)
    return df

In [ ]:
# ==========================================
# 2. 그래프 함수 (x축=layer, y축=metric, method별 선 오버레이)
# ==========================================
def plot_metric(df, metric_col, split, slicing_mode, feature_mode, ax=None):
    sub = df[df["split"] == split]

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    for method_key, label in METHOD_LABELS.items():
        m = sub[sub["method"] == method_key].sort_values("layer")
        if m.empty:
            continue
        style = METHOD_STYLES.get(label, {})
        ax.plot(m["layer"], m[metric_col], label=label, **style)

    ax.set_xlabel("Layer")
    ax.set_ylabel(metric_col)
    ax.set_title(f"{metric_col}\n({slicing_mode}+{feature_mode}, {split} set)")
    ax.grid(alpha=0.3)
    ax.legend()
    return ax

In [ ]:
# ==========================================
# 3. 4개 조합 x 3개 metric 전체 그래프 그리기
# ==========================================
for slicing_mode, feature_mode in COMBOS:
    df = load_distance_csv(slicing_mode, feature_mode)

    for metric in METRICS:
        plot_metric(df, metric, SPLIT_TO_PLOT, slicing_mode, feature_mode)
        plt.tight_layout()
        plt.show()